# NS10 Tutorial C - The Random-Walk / Local-Search Model

**Lecture:** NS10 - Preferential Attachment Extensions

**Reading:** Vázquez (2003), Saramäki and Kaski (2004)

## Learning goals
- Implement the lecture local-search rule:
  1. choose a random target node $j$,
  2. attach first to $j$,
  3. for later edges, choose a neighbor of $j$ with probability $p$, otherwise a random node.
- Understand how local exploration creates **implicit preferential attachment**.
- Measure how the parameter $p$ changes clustering and path length.
- Compare the model to BA.


In [ ]:
from netsci_utils import *
import pandas as pd

set_seeds()
%matplotlib inline


def random_walk_attachment_model(n, m, p, seed=RANDOM_SEED, return_attachment_counts=False):
    rng = np.random.default_rng(seed)
    G = nx.complete_graph(max(m + 1, 3))
    attachment_counts = {node: 0 for node in G.nodes()}

    for new_node in range(G.number_of_nodes(), n):
        existing = list(G.nodes())
        first_target = int(rng.choice(existing))
        targets = {first_target}

        while len(targets) < m:
            if rng.random() < p and len(list(G.neighbors(first_target))) > 0:
                candidates = list(set(G.neighbors(first_target)) - targets)
                if candidates:
                    targets.add(int(rng.choice(candidates)))
                    continue
            fallback = list(set(existing) - targets)
            if not fallback:
                break
            targets.add(int(rng.choice(fallback)))

        G.add_node(new_node)
        attachment_counts[new_node] = 0
        for target in targets:
            G.add_edge(new_node, target)
            attachment_counts[target] += 1

    if return_attachment_counts:
        return G, attachment_counts
    return G


def local_search_selection_frequency(G, p, n_samples=30000, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    nodes = np.array(list(G.nodes()))
    counts = {node: 0 for node in nodes}
    for _ in range(n_samples):
        target = int(rng.choice(nodes))
        endpoint = target
        neighbors = list(G.neighbors(target))
        if neighbors and rng.random() < p:
            endpoint = int(rng.choice(neighbors))
        counts[endpoint] += 1
    total = sum(counts.values())
    return {node: count / total for node, count in counts.items()}

---
## 1. Local closure on a small graph

The difference from BA is visible immediately: after the first edge, later edges can be pulled toward the neighbourhood of the initial target. That creates triangles and local closure.


In [ ]:
rw_small_low = random_walk_attachment_model(25, 2, p=0.0, seed=RANDOM_SEED)
rw_small_high = random_walk_attachment_model(25, 2, p=0.7, seed=RANDOM_SEED)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
pos_low = nx.spring_layout(rw_small_low, seed=RANDOM_SEED)
pos_high = nx.spring_layout(rw_small_high, seed=RANDOM_SEED)
draw_graph(rw_small_low, pos=pos_low, ax=axes[0], with_labels=False, node_size=180)
draw_graph(rw_small_high, pos=pos_high, ax=axes[1], with_labels=False, node_size=180)
axes[0].set_title('p = 0.0: no local walk')
axes[1].set_title('p = 0.7: strong local closure')
plt.show()

print(pd.DataFrame([
    model_summary_row(rw_small_low, 'p = 0.0'),
    model_summary_row(rw_small_high, 'p = 0.7'),
]).round(3).to_string(index=False))


---
## 2. Parameter sweep: clustering versus path length

The lecture claim is that the local-search parameter $p$ raises clustering while keeping paths short.


In [ ]:
p_values = [0.0, 0.2, 0.4, 0.7, 1.0]
sweep_rows = []
sweep_graphs = {}
for p in p_values:
    G = random_walk_attachment_model(1200, 2, p=p, seed=RANDOM_SEED)
    sweep_graphs[p] = G
    row = model_summary_row(G, f'p = {p:.1f}')
    row['p'] = p
    sweep_rows.append(row)

sweep_df = pd.DataFrame(sweep_rows).sort_values('p')
print(sweep_df.round(3).to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(sweep_df['p'], sweep_df['avg clustering'], marker='o', linewidth=2, color=CATEGORY_PALETTE['blue'])
style_axis(axes[0], title='Clustering rises with local search', xlabel='p', ylabel='Average clustering')

axes[1].plot(sweep_df['p'], sweep_df['avg path length in LCC'], marker='o', linewidth=2, color=CATEGORY_PALETTE['orange'])
style_axis(axes[1], title='Paths remain short', xlabel='p', ylabel='Average path length in LCC')

plt.show()


---
## 3. Compare random-walk attachment to BA

We keep the same $n$ and $m$ and compare the model at $p=0.7$ to BA.


In [ ]:
rw_graph = sweep_graphs[0.7]
ba_graph = nx.barabasi_albert_graph(1200, 2, seed=RANDOM_SEED)

compare_df = pd.DataFrame([
    model_summary_row(ba_graph, 'Barabasi-Albert'),
    model_summary_row(rw_graph, 'Random walk / local search (p = 0.7)'),
])
print(compare_df.round(3).to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=FIGURE_SIZE)
plot_ccdf([degree for _, degree in ba_graph.degree()], ax=ax, label='BA', color=CATEGORY_PALETTE['green'], markersize=3)
plot_ccdf([degree for _, degree in rw_graph.degree()], ax=ax, label='Random walk / local search', color=CATEGORY_PALETTE['blue'], markersize=3)
ax.set_title('Degree CCDF: BA versus random-walk attachment')
ax.legend(frameon=False)
plt.show()


---
## 4. Why the rule produces implicit preferential attachment

A local search reaches high-degree nodes more often because high-degree nodes are incident to more edges.

We test that directly by sampling endpoints under the local-search rule on the generated graph and comparing the empirical selection frequency to the normalized degree.


In [ ]:
frequency = local_search_selection_frequency(rw_graph, p=0.7, n_samples=40000, seed=RANDOM_SEED)
degree_dict = dict(rw_graph.degree())
total_degree = sum(degree_dict.values())

frequency_df = pd.DataFrame({
    'node': list(rw_graph.nodes()),
    'degree': [degree_dict[node] for node in rw_graph.nodes()],
    'empirical selection frequency': [frequency[node] for node in rw_graph.nodes()],
    'normalized degree': [degree_dict[node] / total_degree for node in rw_graph.nodes()],
})

fig, ax = plt.subplots(figsize=FIGURE_SIZE_SMALL)
ax.scatter(
    frequency_df['normalized degree'],
    frequency_df['empirical selection frequency'],
    color=CATEGORY_PALETTE['blue'],
    alpha=0.45,
    s=18,
)
lim = max(frequency_df['normalized degree'].max(), frequency_df['empirical selection frequency'].max())
ax.plot([0, lim], [0, lim], linestyle='--', color=CATEGORY_PALETTE['red'], linewidth=2)
style_axis(
    ax,
    title='Selection frequency tracks node degree',
    xlabel='Normalized degree',
    ylabel='Empirical local-search frequency',
)
plt.show()

print(f"Correlation = {frequency_df[['normalized degree', 'empirical selection frequency']].corr().iloc[0, 1]:.3f}")


## Takeaway

- The random-walk model replaces global degree knowledge with **local search**.
- It creates implicit preferential attachment because high-degree nodes are reached more often through neighbours.
- Unlike BA, it also raises clustering by introducing local closure.
- This is the right extension when links are formed through exploration of nearby neighbourhoods rather than global ranking.
